# 08 — Inferencia completa de sentimiento

**Modelo:** RoBERTuito fine-tuned (sliding-window mean, seed 2050)  
**Input:** `df_final_para_modelo.jsonl` — 23 120 intervenciones (1 224 etiquetadas manualmente + 21 896 sin etiqueta)  
**Output:** `df_final_con_sentimiento.jsonl` / `.csv` — mismo dataset con las columnas de predicción agregadas

## Pasos
1. Montar Google Drive y cargar datos
2. Cargar el modelo fine-tuned y el tokenizer
3. Correr inferencia sliding-window (agg=mean) sobre las 23 120 filas
4. Validar contra las 1 224 etiquetas manuales
5. Exportar dataset enriquecido

> ⚠️ **Pre-requisito:** el modelo debe estar guardado en Google Drive (o en `/content/`) con la estructura estándar de Hugging Face (`config.json`, `model.safetensors`, `tokenizer_config.json`, etc.).  
> Si corriste el notebook 06 con `DELETE_OUTPUT_DIR = False`, el modelo quedó en `/content/robertuito_finetuned_parlamento_seed2050/`.

## 0. Instalar dependencias (solo Colab)

In [ ]:
# Ejecutar solo si estás en Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q transformers datasets torch tqdm pandas

## 1. Montar Google Drive (solo Colab)

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

## 2. Configuración de rutas

Ajustá `MODEL_PATH` y `DATA_PATH` según dónde quedó el modelo y el dataset.

In [ ]:
from pathlib import Path

# ── Ruta al modelo fine-tuned (Hugging Face format) ───────────────────────────
# Opción A: modelo guardado en Google Drive
MODEL_PATH = Path("/content/drive/MyDrive/Tesis/robertuito_finetuned_parlamento_seed2050")

# Opción B: modelo guardado localmente en Colab (si no se borró)
# MODEL_PATH = Path("/content/robertuito_finetuned_parlamento_seed2050")

# ── Ruta al dataset completo ───────────────────────────────────────────────────
DATA_PATH   = Path("/content/df_final_para_modelo.jsonl")

# ── Salidas ───────────────────────────────────────────────────────────────────
OUT_JSONL   = Path("/content/df_final_con_sentimiento.jsonl")
OUT_CSV     = Path("/content/df_final_con_sentimiento.csv")

# ── Hiperparámetros de inferencia (deben coincidir con el entrenamiento) ───────
MAX_LEN_SW  = 128
STRIDE_SW   = 96
BATCH_SIZE  = 32    # intervenciones por batch (ajustá según VRAM disponible)

# Verificar que el modelo existe
assert MODEL_PATH.exists(), f"❌ No se encontró el modelo en {MODEL_PATH}"
assert DATA_PATH.exists(),  f"❌ No se encontró el dataset en {DATA_PATH}"
print(f"✅ Modelo: {MODEL_PATH}")
print(f"✅ Dataset: {DATA_PATH}")

## 3. Imports y utilidades

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

# ── Mapeo de etiquetas ────────────────────────────────────────────────────────
# Debe coincidir con el label2id usado en el entrenamiento
LABEL2ID = {"N": 0, "Neu": 1, "P": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 4. Función de inferencia sliding-window

In [ ]:
def softmax_stable(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Softmax numéricamente estable."""
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


@torch.no_grad()
def predict_proba_windows(
    text: str,
    model: AutoModelForSequenceClassification,
    tokenizer: AutoTokenizer,
    max_len: int = 128,
    stride: int = 96,
    device: torch.device = None,
) -> np.ndarray:
    """
    Tokeniza `text` con ventana deslizante y devuelve la probabilidad
    promedio de cada clase sobre todas las ventanas.

    Returns
    -------
    probs : np.ndarray, shape (n_classes,)
    """
    if device is None:
        device = next(model.parameters()).device

    model.eval()

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids_list   = enc["input_ids"]
    attention_mask_list = enc["attention_mask"]

    # Padding manual al max de la ventana
    max_len_enc = max(len(ids) for ids in input_ids_list)
    pad_id = tokenizer.pad_token_id

    padded_ids  = [ids + [pad_id] * (max_len_enc - len(ids)) for ids in input_ids_list]
    padded_mask = [msk + [0]      * (max_len_enc - len(msk)) for msk in attention_mask_list]

    ids_tensor  = torch.tensor(padded_ids,  dtype=torch.long).to(device)
    mask_tensor = torch.tensor(padded_mask, dtype=torch.long).to(device)

    outputs = model(input_ids=ids_tensor, attention_mask=mask_tensor)
    logits  = outputs.logits.cpu().numpy()          # (n_ventanas, n_classes)
    probs   = softmax_stable(logits, axis=-1)        # (n_ventanas, n_classes)

    return probs.mean(axis=0)                        # (n_classes,)


def predict_text_sliding(
    text: str,
    model: AutoModelForSequenceClassification,
    tokenizer: AutoTokenizer,
    id2label: dict,
    max_len: int = 128,
    stride: int = 96,
    device: torch.device = None,
) -> dict:
    """
    Predice el sentimiento de `text` con sliding-window (agg=mean).

    Returns
    -------
    dict con:
        sentimiento_pred : str   ("N", "Neu", "P")
        pred_conf        : float (probabilidad de la clase ganadora)
        prob_N           : float
        prob_Neu         : float
        prob_P           : float
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return {
            "sentimiento_pred": None,
            "pred_conf": None,
            "prob_N": None,
            "prob_Neu": None,
            "prob_P": None,
        }

    probs      = predict_proba_windows(text, model, tokenizer, max_len, stride, device)
    pred_id    = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    pred_conf  = float(probs[pred_id])

    return {
        "sentimiento_pred": pred_label,
        "pred_conf":  pred_conf,
        "prob_N":     float(probs[LABEL2ID["N"]]),
        "prob_Neu":   float(probs[LABEL2ID["Neu"]]),
        "prob_P":     float(probs[LABEL2ID["P"]]),
    }

## 5. Cargar modelo y tokenizer

In [ ]:
print(f"Cargando modelo desde: {MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model = model.to(device)
model.eval()

print(f"✅ Modelo cargado. Parámetros: {sum(p.numel() for p in model.parameters()):,}")

## 6. Cargar dataset

In [ ]:
df = pd.read_json(DATA_PATH, lines=True)
print(f"Filas totales: {len(df):,}")
print(f"Columnas: {df.columns.tolist()}")
print()

# Resumen de etiquetas manuales
if "sentimiento" in df.columns:
    etiquetados = df["sentimiento"].notna().sum()
    sin_etiqueta = df["sentimiento"].isna().sum()
    print(f"Con etiqueta manual : {etiquetados:,}")
    print(f"Sin etiqueta        : {sin_etiqueta:,}")
    print()
    print("Distribución etiquetas manuales:")
    print(df["sentimiento"].value_counts(dropna=False))

## 7. Inferencia sobre todo el dataset

Se procesa de a `BATCH_SIZE` textos para mostrar progreso. El sliding-window corre por texto de forma individual (para manejar longitudes variables).

In [ ]:
resultados = []

textos = df["intervencion"].tolist()

for texto in tqdm(textos, desc="Inferencia sliding-window", unit="interv."):
    resultado = predict_text_sliding(
        texto,
        model,
        tokenizer,
        id2label=ID2LABEL,
        max_len=MAX_LEN_SW,
        stride=STRIDE_SW,
        device=device,
    )
    resultados.append(resultado)

df_pred = pd.DataFrame(resultados)
print(f"✅ Inferencia completada. Shape: {df_pred.shape}")
df_pred.head()

## 8. Combinar con el dataset original

In [ ]:
df_out = pd.concat([df.reset_index(drop=True), df_pred], axis=1)

print(f"Distribución de predicciones (todo el corpus):")
print(df_out["sentimiento_pred"].value_counts())
print()
print(f"Confianza promedio: {df_out['pred_conf'].mean():.4f}")
df_out[["intervencion", "sentimiento", "sentimiento_pred", "pred_conf"]].head(10)

## 9. Validación contra etiquetas manuales

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Filtrar solo filas con etiqueta manual
df_eval = df_out[df_out["sentimiento"].notna()].copy()
print(f"Filas con etiqueta manual: {len(df_eval):,}")

y_true = df_eval["sentimiento"].tolist()
y_pred = df_eval["sentimiento_pred"].tolist()

print("\n── Classification Report ──────────────────────────────────────")
print(classification_report(y_true, y_pred, labels=["N", "Neu", "P"], digits=4))

print("── Confusion Matrix (filas=real, columnas=pred) ───────────────")
cm = confusion_matrix(y_true, y_pred, labels=["N", "Neu", "P"])
df_cm = pd.DataFrame(cm, index=["Real_N", "Real_Neu", "Real_P"],
                          columns=["Pred_N", "Pred_Neu", "Pred_P"])
print(df_cm)

## 10. Exportar dataset enriquecido

In [ ]:
# ── JSONL ─────────────────────────────────────────────────────────────────────
df_out.to_json(OUT_JSONL, orient="records", lines=True, force_ascii=False)
print(f"✅ Guardado JSONL : {OUT_JSONL}  ({OUT_JSONL.stat().st_size / 1e6:.1f} MB)")

# ── CSV ───────────────────────────────────────────────────────────────────────
df_out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"✅ Guardado CSV   : {OUT_CSV}  ({OUT_CSV.stat().st_size / 1e6:.1f} MB)")

## 11. (Opcional) Copiar a Google Drive

In [ ]:
import shutil

DRIVE_OUT = Path("/content/drive/MyDrive/Tesis/data/processed")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

shutil.copy(OUT_JSONL, DRIVE_OUT / OUT_JSONL.name)
shutil.copy(OUT_CSV,   DRIVE_OUT / OUT_CSV.name)

print(f"✅ Archivos copiados a {DRIVE_OUT}")

---
## Columnas agregadas al dataset de salida

| Columna | Tipo | Descripción |
|---|---|---|
| `sentimiento_pred` | `str` | Predicción del modelo (`N`, `Neu`, `P`) |
| `pred_conf` | `float` | Probabilidad de la clase predicha (0–1) |
| `prob_N` | `float` | Probabilidad de clase Negativo |
| `prob_Neu` | `float` | Probabilidad de clase Neutro |
| `prob_P` | `float` | Probabilidad de clase Positivo |

> La columna `sentimiento` (etiqueta manual) se conserva intacta. En las filas sin etiqueta manual queda en `NaN`.